## SILVER LAYER - CLEANED

In [0]:
%run ./00_config

### Create Table Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyc_taxi_project.silver.yellow_taxi_silver
(
  VendorID                INT,
  tpep_pickup_datetime    TIMESTAMP,
  tpep_dropoff_datetime   TIMESTAMP,
  passenger_count         INT,
  trip_distance           DOUBLE,
  PULocationID            INT,
  DOLocationID            INT,
  payment_type            INT,
  fare_amount             DOUBLE,
  tip_amount              DOUBLE,
  tolls_amount            DOUBLE,
  total_amount            DOUBLE,
  congestion_surcharge    DOUBLE,
  -- Derived columns
  pickup_year             INT,
  pickup_month            INT,
  pickup_day              INT,
  pickup_hour             INT,
  trip_duration_minutes   DOUBLE,
  -- Metadata
  ingested_at             TIMESTAMP,
  source_file             STRING
)
USING DELTA
COMMENT 'Silver layer - cleaned NYC Yellow Taxi data'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');

### Transform And Clean Functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def transform_silver(df):

    # ── 0. Parse _rescued_data để lấy lại các cột bị mất ──────
    df = (df
        .withColumn("_rescued_json", F.col("_rescued_data").cast("string"))

        # Lấy giá trị từ _rescued_data nếu cột gốc bị null
        .withColumn("VendorID",
            F.coalesce(
                F.col("VendorID"),
                F.get_json_object(F.col("_rescued_json"), "$.VendorID").cast("long")
            ))
        .withColumn("PULocationID",
            F.coalesce(
                F.col("PULocationID"),
                F.get_json_object(F.col("_rescued_json"), "$.PULocationID").cast("long")
            ))
        .withColumn("DOLocationID",
            F.coalesce(
                F.col("DOLocationID"),
                F.get_json_object(F.col("_rescued_json"), "$.DOLocationID").cast("long")
            ))
        .withColumn("passenger_count",
            F.coalesce(
                F.col("passenger_count"),
                F.get_json_object(F.col("_rescued_json"), "$.passenger_count").cast("double")
            ))
        .withColumn("RatecodeID",
            F.coalesce(
                F.col("RatecodeID"),
                F.get_json_object(F.col("_rescued_json"), "$.RatecodeID").cast("double")
            ))
        .drop("_rescued_json")
    )

    # ── 1. Drop null ở các cột quan trọng ──────────────────────
    df = (df
        .filter(F.col("tpep_pickup_datetime").isNotNull())
        .filter(F.col("tpep_dropoff_datetime").isNotNull())
        .filter(F.col("total_amount").isNotNull())

        # ── 2. Fill null ở các cột ít quan trọng hơn ───────────
        .fillna({
            "VendorID"            : 0,
            "passenger_count"     : 1,
            "congestion_surcharge": 0.0,
            "tolls_amount"        : 0.0,
            "tip_amount"          : 0.0,
        })

        # ── 3. Lọc outliers ────────────────────────────────────
        .filter(F.col("trip_distance")   >  0)
        .filter(F.col("fare_amount")     >  0)
        .filter(F.col("total_amount")    >  0)
        .filter(F.col("passenger_count") >  0)
        .filter(F.col("passenger_count") <= 6)
        .filter(F.col("trip_distance")   <= 200)

        # ── 4. Lọc datetime không hợp lệ ───────────────────────
        .filter(F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"))
        .filter(F.year("tpep_pickup_datetime").between(2019, 2025))

        # ── 5. Tạo derived columns ──────────────────────────────
        .withColumn("pickup_year",
            F.year("tpep_pickup_datetime"))
        .withColumn("pickup_month",
            F.month("tpep_pickup_datetime"))
        .withColumn("pickup_day",
            F.dayofmonth("tpep_pickup_datetime"))
        .withColumn("pickup_hour",
            F.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_minutes",
            F.round(
                (F.unix_timestamp("tpep_dropoff_datetime") -
                 F.unix_timestamp("tpep_pickup_datetime")) / 60,
            2))

        # ── 6. Cast kiểu dữ liệu ───────────────────────────────
        .withColumn("VendorID",        F.col("VendorID").cast("int"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("PULocationID",    F.col("PULocationID").cast("int"))
        .withColumn("DOLocationID",    F.col("DOLocationID").cast("int"))
        .withColumn("payment_type",    F.col("payment_type").cast("int"))

        # ── 7. Chỉ giữ các cột cần thiết ──────────────────────
        .select(
            "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "passenger_count", "trip_distance", "PULocationID", "DOLocationID",
            "payment_type", "fare_amount", "tip_amount", "tolls_amount",
            "total_amount", "congestion_surcharge",
            "pickup_year", "pickup_month", "pickup_day",
            "pickup_hour", "trip_duration_minutes",
            "ingested_at", "source_file"
        )
    )

    return df

### Process Duplicate

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batchId):

    window = Window.partitionBy(
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime" 
    ).orderBy(F.col("ingested_at").desc())

    (microBatchDF
        .withColumn("rank", F.rank().over(window))
        .filter(F.col("rank") == 1)
        .drop("rank")
        .createOrReplaceTempView("silver_updates"))

    query = """
        MERGE INTO nyc_taxi_project.silver.yellow_taxi_silver t
        USING silver_updates s
        ON  t.VendorID              = s.VendorID
        AND t.tpep_pickup_datetime  = s.tpep_pickup_datetime
        AND t.tpep_dropoff_datetime = s.tpep_dropoff_datetime

            WHEN MATCHED AND t.ingested_at < s.ingested_at
                THEN UPDATE SET *

            WHEN NOT MATCHED
                THEN INSERT *
    """

    microBatchDF.sparkSession.sql(query)

### Silver Streaming

In [0]:
def process_silver_cleaned():  
   query = (spark.readStream
               .format("delta")
               .option("ignoreDeletes", "true")
               .table(BRONZE_TABLE)
               .transform(transform_silver)
            .writeStream
               .foreachBatch(batch_upsert)
               .option("checkpointLocation", SILVER1_CHECKPOINT)
               .trigger(availableNow=True))
   query.start().awaitTermination()
process_silver_cleaned() 

In [0]:
%sql
ALTER TABLE nyc_taxi_project.silver.yellow_taxi_silver 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

### Check Data Loss

In [0]:
%sql
SELECT
  'Bronze' AS layer,  COUNT(*) AS total_rows
FROM nyc_taxi_project.bronze.yellow_taxi

UNION ALL

SELECT
  'Silver' AS layer,  COUNT(*) AS total_rows
FROM nyc_taxi_project.silver.yellow_taxi_silver;